In [ ]:
pip install azure-ai-documentintelligence azure-cognitiveservices-vision-face azure-storage-blob python-dotenv msrest pillow opencv-python flask

In [4]:
import os
from dotenv import load_dotenv
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.core.credentials import AzureKeyCredential
from azure.cognitiveservices.vision.face import FaceClient
from msrest.authentication import CognitiveServicesCredentials
from azure.storage.blob import BlobServiceClient

load_dotenv()

with open(r".\.env") as f:
    for line in f:
        if line.startswith("AZURE_STORAGE"):
            key, val = line.strip().split("=", 1)
            os.environ[key] = val
            print(f"Cargado: {key} = {val[:40]}...")

# Test Document Intelligence
doc_client = DocumentIntelligenceClient(
    endpoint=os.getenv("AZURE_DOC_INTELLIGENCE_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("AZURE_DOC_INTELLIGENCE_KEY"))
)
print("Document Intelligence: conectado")

# Test Face API
face_client = FaceClient(
    os.getenv("AZURE_FACE_ENDPOINT"),
    CognitiveServicesCredentials(os.getenv("AZURE_FACE_KEY"))
)
print("Face API: conectado")

# Test Blob Storage
blob_client = BlobServiceClient.from_connection_string(
    os.getenv("AZURE_STORAGE_CONNECTION_STRING")
)
containers = [c.name for c in blob_client.list_containers()]
print(f"Blob Storage: contenedores encontrados → {containers}")

Cargado: AZURE_STORAGE_CONNECTION_STRING = DefaultEndpointsProtocol=https;EndpointS...
Document Intelligence: conectado
Face API: conectado
Blob Storage: contenedores encontrados → ['documentos', 'resultados', 'scm-releases', 'selfies']


In [1]:
# preprocesar.py
import os
from PIL import Image

def preprocesar(ruta_entrada, ruta_salida):
    img = Image.open(ruta_entrada)

    if img.mode != "RGB":
        img = img.convert("RGB")

    max_ancho = 1200
    if img.width > max_ancho:
        ratio = max_ancho / img.width
        img = img.resize((max_ancho, int(img.height * ratio)), Image.LANCZOS)

    img.save(ruta_salida, "PNG", optimize=True)
    print(f"{os.path.basename(ruta_salida)} — {img.width}x{img.height}px")

os.makedirs("datos_procesados", exist_ok=True)

for archivo in os.listdir("datos_crudos"):
    if archivo.lower().endswith((".png", ".jpg", ".jpeg")):
        nombre_salida = os.path.splitext(archivo)[0] + ".png"
        preprocesar(
            os.path.join("datos_crudos", archivo),
            os.path.join("datos_procesados", nombre_salida)
        )

print(f"\nTotal: {len(os.listdir('datos_procesados'))} imágenes procesadas")

onb_sol_000001_doc.png — 1200x849px
onb_sol_000001_selfie.png — 880x1215px

Total: 2 imágenes procesadas


In [2]:
# subir_datos.py
import os
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()
servicio = BlobServiceClient.from_connection_string(
    os.getenv("AZURE_STORAGE_CONNECTION_STRING")
)

def subir(ruta_local, contenedor):
    nombre = os.path.basename(ruta_local)
    cliente = servicio.get_container_client(contenedor)
    with open(ruta_local, "rb") as f:
        cliente.upload_blob(nombre, f, overwrite=True)
    print(f"{nombre} → {contenedor}/")

for archivo in os.listdir("datos_procesados"):
    ruta = os.path.join("datos_procesados", archivo)
    if "_doc.png" in archivo:
        subir(ruta, "documentos")
    elif "_selfie.png" in archivo:
        subir(ruta, "selfies")

print("\nIngesta completa")

onb_sol_000001_doc.png → documentos/
onb_sol_000001_selfie.png → selfies/

Ingesta completa


In [3]:
# validar_extraccion.py
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient, generate_blob_sas, BlobSasPermissions

load_dotenv(r"C:\Users\jhona\OneDrive\Documentos\GitHub\ProjectWithAzure-VerificationOfKyc\.env", override=True)

CUENTA = "kycstoragedev2"

# ── clientes ──────────────────────────────────────────────
doc_cliente = DocumentIntelligenceClient(
    endpoint=os.getenv("AZURE_DOC_INTELLIGENCE_ENDPOINT"),
    credential=AzureKeyCredential(os.getenv("AZURE_DOC_INTELLIGENCE_KEY"))
)

blob_service = BlobServiceClient.from_connection_string(
    os.getenv("AZURE_STORAGE_CONNECTION_STRING")
)

# ── detectar solicitudes disponibles ─────────────────────
blobs = blob_service.get_container_client("documentos").list_blobs()
solicitudes = sorted([
    b.name.replace("onb_sol_", "").replace("_doc.png", "")
    for b in blobs
    if b.name.endswith("_doc.png")
])

print(f"Solicitudes encontradas: {solicitudes}\n")

if not solicitudes:
    print("No hay imágenes en el contenedor 'documentos'. Sube archivos primero.")
    exit()

# ── función SAS ───────────────────────────────────────────
def generar_url_sas(contenedor, blob_name):
    sas = generate_blob_sas(
        account_name=CUENTA,
        container_name=contenedor,
        blob_name=blob_name,
        account_key=os.getenv("AZURE_STORAGE_KEY"),
        permission=BlobSasPermissions(read=True),
        expiry=datetime.now(timezone.utc) + timedelta(hours=1)
    )
    return f"https://{CUENTA}.blob.core.windows.net/{contenedor}/{blob_name}?{sas}"

# ── procesar cada solicitud ───────────────────────────────
for sid in solicitudes:
    print(f"Procesando solicitud {sid}...")
    try:
        url = generar_url_sas("documentos", f"onb_sol_{sid}_doc.png")

        poller    = doc_cliente.begin_analyze_document(
            "prebuilt-idDocument",
            AnalyzeDocumentRequest(url_source=url)
        )
        resultado = poller.result()

        for doc in resultado.documents:
            campos = {}
            for campo, valor in doc.fields.items():
                if valor and valor.content:
                    campos[campo] = {
                        "valor":     valor.content,
                        "confianza": round(valor.confidence, 3)
                    }
            print(f"  Campos extraídos:")
            for k, v in campos.items():
                print(f"    {k:20s}: {v['valor']:30s} (confianza: {v['confianza']})")

    except Exception as e:
        print(f"  Error en {sid}: {e}")

print("\n Validación completada")

Solicitudes encontradas: ['000001']

Procesando solicitud 000001...
  Campos extraídos:
    DateOfBirth         : 04 MARZO 1980                  (confianza: 0.855)
    DateOfExpiration    : 04 MARZO 2024                  (confianza: 0.709)
    DocumentNumber      : 080-0006236-7                  (confianza: 0.824)
    FirstName           : ORDANEX ANGNIOLIS              (confianza: 0.556)
    LastName            : ROSELLO FELIZ                  (confianza: 0.652)

 Validación completada


In [4]:
#Validacion de FACE API
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
from azure.cognitiveservices.vision.face import FaceClient
from msrest.authentication import CognitiveServicesCredentials
from azure.storage.blob import BlobServiceClient, generate_blob_sas, BlobSasPermissions

load_dotenv(r"C:\Users\jhona\OneDrive\Documentos\GitHub\ProjectWithAzure-VerificationOfKyc\.env", override=True)

CUENTA = "kycstoragedev2"

cliente = FaceClient(
    "https://centralus.api.cognitive.microsoft.com/",
    CognitiveServicesCredentials(os.getenv("AZURE_FACE_KEY"))
)

blob_service = BlobServiceClient.from_connection_string(
    os.getenv("AZURE_STORAGE_CONNECTION_STRING")
)

# ── solicitudes disponibles ───────────────────────────────
blobs = blob_service.get_container_client("documentos").list_blobs()
solicitudes = sorted([
    b.name.replace("onb_sol_", "").replace("_doc.png", "")
    for b in blobs if b.name.endswith("_doc.png")
])
print(f"Solicitudes encontradas: {solicitudes}\n")

# ── SAS ───────────────────────────────────────────────────
def generar_url_sas(contenedor, blob_name):
    sas = generate_blob_sas(
        account_name=CUENTA,
        container_name=contenedor,
        blob_name=blob_name,
        account_key=os.getenv("AZURE_STORAGE_KEY"),
        permission=BlobSasPermissions(read=True),
        expiry=datetime.now(timezone.utc) + timedelta(hours=1)
    )
    return f"https://{CUENTA}.blob.core.windows.net/{contenedor}/{blob_name}?{sas}"

# ── detectar con landmarks ────────────────────────────────
def detectar(url, nombre):
    rostros = cliente.face.detect_with_url(
        url,
        detection_model="detection_01",
        return_face_id=False,
        return_face_landmarks=True,
        return_face_attributes=["headPose", "blur", "exposure"]
    )
    if not rostros:
        raise ValueError(f"No se detectó rostro en {nombre}")
    print(f"  ✓ Rostro detectado en {nombre}")
    return rostros[0]

# ── calcular confidence por landmarks ────────────────────
def calcular_confidence(face1, face2):
    def landmarks_a_vector(face):
        lm = face.face_landmarks
        puntos = [
            lm.pupil_left, lm.pupil_right,
            lm.nose_tip,
            lm.mouth_left, lm.mouth_right,
            lm.eyebrow_left_outer, lm.eyebrow_right_outer,
            lm.under_lip_bottom
        ]
        return [(p.x, p.y) for p in puntos]

    def normalizar(puntos):
        xs = [p[0] for p in puntos]
        ys = [p[1] for p in puntos]
        cx = sum(xs) / len(xs)
        cy = sum(ys) / len(ys)
        escala = max(max(xs) - min(xs), max(ys) - min(ys))
        if escala == 0:
            return puntos
        return [((p[0] - cx) / escala, (p[1] - cy) / escala) for p in puntos]

    v1 = normalizar(landmarks_a_vector(face1))
    v2 = normalizar(landmarks_a_vector(face2))

    distancia = sum(
        ((a[0] - b[0])**2 + (a[1] - b[1])**2)**0.5
        for a, b in zip(v1, v2)
    ) / len(v1)

    confidence = round(max(0.0, min(1.0, 1 - distancia * 2)), 4)
    return confidence

# ── pipeline por solicitud ────────────────────────────────
def procesar(sid):
    url_doc    = generar_url_sas("documentos", f"onb_sol_{sid}_doc.png")
    url_selfie = generar_url_sas("selfies",    f"onb_sol_{sid}_selfie.png")

    face_doc    = detectar(url_doc,    "documento")
    face_selfie = detectar(url_selfie, "selfie")

    confidence = calcular_confidence(face_doc, face_selfie)

    if confidence >= 0.90:
        clasificacion = "aprobada_automatica"
    elif confidence >= 0.70:
        clasificacion = "revision_manual_requerida"
    else:
        clasificacion = "rechazada"

    print(f"  confidence   : {confidence}")
    print(f"  clasificacion: {clasificacion}")
    return clasificacion

# ── main ──────────────────────────────────────────────────
for sid in solicitudes:
    print(f"\nProcesando solicitud {sid}...")
    try:
        procesar(sid)
    except Exception as e:
        print(f"  Error en {sid}: {e}")

print("\n✓ Validación facial completada")

Solicitudes encontradas: ['000001']


Procesando solicitud 000001...
  ✓ Rostro detectado en documento
  ✓ Rostro detectado en selfie
  confidence   : 0.9423
  clasificacion: aprobada_automatica

✓ Validación facial completada
